# Vision Training: Model Pool and Prediction Generation

Trains the fixed 15-model vision pool (ImageNet-pretrained CNNs and transformers,
each with its head replaced by a K-way linear layer) on one dataset (here
CIFAR-10) and writes the prediction JSON consumed by the experiment pipeline.

Training uses the official train/test splits where available; sizing follows the
paper (N_test_min = ceil(20/p_min), N_test_target = 10 N_test_min); validation is
used only for checkpoint selection. Output schema matches src/io_utils.py.


In [ ]:
# ============================================================
# VISION POOL v1 (15 models) — load all + replace head to num_classes
# TorchVision-first; optional timm fallback for missing transformers.
# One-cell sanity check before building full notebook.
# ============================================================

import torch
import torch.nn as nn

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
num_classes = 10  # <-- set per dataset later

# ----------------------------
# TorchVision imports
# ----------------------------
import torchvision
import torchvision.models as tvm

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("device:", DEVICE)

# ----------------------------
# Optional timm fallback
# ----------------------------
try:
    import timm
    HAS_TIMM = True
    print("timm:", timm.__version__)
except Exception as e:
    HAS_TIMM = False
    print("timm: not available (ok if torchvision has all you need)")

# ----------------------------
# Define the pool (TorchVision-first IDs + optional timm IDs)
# ----------------------------
# CNN / Mobile / Modern CNN: usually available in torchvision
VISION_POOL = [
    # Group A — ResNets
    {"name": "resnet18",         "src": "torchvision", "tv_fn": "resnet18",         "weights": "DEFAULT"},
    {"name": "resnet34",         "src": "torchvision", "tv_fn": "resnet34",         "weights": "DEFAULT"},
    {"name": "resnet50",         "src": "torchvision", "tv_fn": "resnet50",         "weights": "DEFAULT"},
    {"name": "resnet101",        "src": "torchvision", "tv_fn": "resnet101",        "weights": "DEFAULT"},

    # Group B — EfficientNet / ConvNeXt
    {"name": "efficientnet_b0",  "src": "torchvision", "tv_fn": "efficientnet_b0",  "weights": "DEFAULT"},
    {"name": "efficientnet_b2",  "src": "torchvision", "tv_fn": "efficientnet_b2",  "weights": "DEFAULT"},
    {"name": "convnext_tiny",    "src": "torchvision", "tv_fn": "convnext_tiny",    "weights": "DEFAULT"},
    {"name": "convnext_small",   "src": "torchvision", "tv_fn": "convnext_small",   "weights": "DEFAULT"},

    # Group C — Mobile/lightweight
    {"name": "mobilenet_v3_large","src": "torchvision","tv_fn": "mobilenet_v3_large","weights": "DEFAULT"},
    {"name": "mobilenet_v2",     "src": "torchvision", "tv_fn": "mobilenet_v2",     "weights": "DEFAULT"},
    {"name": "regnet_y_400mf",   "src": "torchvision", "tv_fn": "regnet_y_400mf",   "weights": "DEFAULT"},
    {"name": "shufflenet_v2_x1_0","src":"torchvision", "tv_fn": "shufflenet_v2_x1_0","weights":"DEFAULT"},

    # Group D — Transformers (TorchVision if available; else timm fallback)
    # TorchVision names vary by version; we attempt tv first, then timm.
    {"name": "vit_b_16",         "src": "auto", "tv_fn": "vit_b_16",   "weights": "DEFAULT",
                                 "timm_id": "vit_base_patch16_224"},
    {"name": "swin_t",           "src": "auto", "tv_fn": "swin_t",     "weights": "DEFAULT",
                                 "timm_id": "swin_tiny_patch4_window7_224"},
    {"name": "deit_s",           "src": "timm_only", "timm_id": "deit_small_patch16_224"},
]

# ----------------------------
# Helper: replace classification head to num_classes (uniform Linear head)
# Works for torchvision + timm (most common cases).
# ----------------------------
def _replace_head(model: nn.Module, num_classes: int) -> None:
    # ResNet/RegNet etc.
    if hasattr(model, "fc") and isinstance(model.fc, nn.Module):
        in_f = model.fc.in_features
        model.fc = nn.Linear(in_f, num_classes)
        return

    # EfficientNet / MobileNetV2 / ConvNeXt etc.
    if hasattr(model, "classifier") and isinstance(model.classifier, nn.Module):
        if isinstance(model.classifier, nn.Sequential):
            for i in range(len(model.classifier)-1, -1, -1):
                if isinstance(model.classifier[i], nn.Linear):
                    in_f = model.classifier[i].in_features
                    model.classifier[i] = nn.Linear(in_f, num_classes)
                    return
        elif isinstance(model.classifier, nn.Linear):
            in_f = model.classifier.in_features
            model.classifier = nn.Linear(in_f, num_classes)
            return

    # Some models expose .head as Linear
    if hasattr(model, "head") and isinstance(model.head, nn.Module):
        if isinstance(model.head, nn.Linear):
            in_f = model.head.in_features
            model.head = nn.Linear(in_f, num_classes)
            return

    # TorchVision ViT often uses .heads (Module / Sequential)
    if hasattr(model, "heads") and isinstance(model.heads, nn.Module):
        # If heads is a Sequential, replace last Linear
        if isinstance(model.heads, nn.Sequential):
            for i in range(len(model.heads)-1, -1, -1):
                if isinstance(model.heads[i], nn.Linear):
                    in_f = model.heads[i].in_features
                    model.heads[i] = nn.Linear(in_f, num_classes)
                    return
        # If heads is a single Linear
        if isinstance(model.heads, nn.Linear):
            in_f = model.heads.in_features
            model.heads = nn.Linear(in_f, num_classes)
            return
        # If heads has a "head" attribute
        if hasattr(model.heads, "head") and isinstance(model.heads.head, nn.Linear):
            in_f = model.heads.head.in_features
            model.heads.head = nn.Linear(in_f, num_classes)
            return

    # timm preferred API
    if hasattr(model, "reset_classifier") and hasattr(model, "get_classifier"):
        model.reset_classifier(num_classes=num_classes)
        return

    raise RuntimeError("Could not find a known classifier head attribute to replace.")


def _load_torchvision(tv_fn: str, weights: str):
    fn = getattr(tvm, tv_fn)  # may raise AttributeError if not available
    if weights == "DEFAULT":
        # torchvision provides Weights enums via .Weights.DEFAULT
        # e.g., tvm.ResNet50_Weights.DEFAULT, but accessible as <fn>_Weights sometimes.
        # The clean way is weights="DEFAULT" using the model's Weights enum if exposed:
        # Newer torchvision supports weights="DEFAULT" directly.
        try:
            return fn(weights="DEFAULT")
        except TypeError:
            # Older versions require explicit weights enum; try to locate it
            weights_enum_name = f"{tv_fn.upper()}_Weights"
            if hasattr(tvm, weights_enum_name):
                W = getattr(tvm, weights_enum_name).DEFAULT
                return fn(weights=W)
            # fallback: no pretrained
            return fn(weights=None)
    else:
        return fn(weights=None)

def _load_timm(timm_id: str):
    if not HAS_TIMM:
        raise RuntimeError("timm not installed, cannot load timm model: " + timm_id)
    return timm.create_model(timm_id, pretrained=True)

# ----------------------------
# Load pool
# ----------------------------
models = {}
report = []

for spec in VISION_POOL:
    name = spec["name"]
    try:
        if spec.get("src") in ("torchvision", "auto"):
            try:
                m = _load_torchvision(spec["tv_fn"], spec.get("weights", "DEFAULT"))
                src_used = f"torchvision.{spec['tv_fn']}"
            except Exception as tv_e:
                # if auto and has timm_id, try timm
                if spec.get("src") == "auto" and spec.get("timm_id") is not None:
                    m = _load_timm(spec["timm_id"])
                    src_used = f"timm.{spec['timm_id']}"
                else:
                    raise tv_e
        elif spec.get("src") == "timm_only":
            m = _load_timm(spec["timm_id"])
            src_used = f"timm.{spec['timm_id']}"
        else:
            raise RuntimeError("Unknown src policy in spec: " + str(spec))

        _replace_head(m, num_classes)
        m.to(DEVICE)
        m.eval()

        # quick check: forward shape with dummy input
        with torch.no_grad():
            x = torch.randn(2, 3, 224, 224, device=DEVICE)
            y = m(x)
            assert y.shape == (2, num_classes), f"Bad output shape: {y.shape}"

        models[name] = m
        report.append((name, "OK", src_used))
    except Exception as e:
        report.append((name, "FAIL", f"{type(e).__name__}: {e}"))

# ----------------------------
# Print report
# ----------------------------
ok = [r for r in report if r[1] == "OK"]
fail = [r for r in report if r[1] == "FAIL"]

print("\n=== VISION POOL LOAD REPORT ===")
print(f"OK: {len(ok)} / {len(report)}")
for name, status, msg in ok:
    print(f"[{status}] {name:18s} <- {msg}")

if fail:
    print(f"\nFAIL: {len(fail)}")
    for name, status, msg in fail:
        print(f"[{status}] {name:18s} :: {msg}")

print("\nLoaded model keys:", list(models.keys()))


In [ ]:
# ============================================================
# VISION DATASET — RATIONALITY CHECK ONLY
# Dataset: CIFAR-10
# Resolution policy: 224x224 (applied later in training)
# ============================================================

import numpy as np
from torchvision.datasets import CIFAR10

# ----------------------------
# Load dataset (train split only)
# ----------------------------
dataset = CIFAR10(
    root="./data",
    train=True,
    download=True
)

# ----------------------------
# Extract labels
# ----------------------------
y_raw = np.array(dataset.targets)
N_total = len(y_raw)

labels, counts = np.unique(y_raw, return_counts=True)
p_min = counts.min() / N_total

# ----------------------------
# Compute size constraints
# ----------------------------
N_test_min = int(np.ceil(20 / p_min))
N_test_target = 10 * N_test_min

N_train_min = int(np.ceil(50 / p_min))
N_train_max = int(np.ceil(500 / p_min))

N_train = min(N_train_max, N_total - N_test_target)

# ----------------------------
# Print summary
# ----------------------------
print("=== DATASET SUMMARY ===")
print("Dataset: CIFAR-10 (train split)")
print(f"Total samples (N): {N_total}")
print(f"Classes: {dict(zip(labels.tolist(), counts.tolist()))}")
print(f"p_min: {p_min:.6f}")
print()

print("=== SIZE CONSTRAINTS ===")
print(f"N_test_min     = ceil(20 / p_min)  = {N_test_min}")
print(f"N_test_target  = 10 * N_test_min   = {N_test_target}")
print(f"N_train_min    = ceil(50 / p_min)  = {N_train_min}")
print(f"N_train_max    = ceil(500 / p_min) = {N_train_max}")
print(f"Chosen N_train = min(N_train_max, N - N_test_target) = {N_train}")
print()

# ----------------------------
# Rationality check
# ----------------------------
if N_train > N_train_min and (N_train + N_test_target) <= N_total:
    print("✅ DATASET PASSES nTrain / nTest RATIONALITY CHECK")
else:
    print("❌ DATASET FAILS nTrain / nTest RATIONALITY CHECK")


In [ ]:
# ============================================================
# VISION CELL 3 — SPLIT + TRANSFORMS (OFFICIAL CIFAR-10 TEST)
# Dataset: CIFAR-10
#
# Protocol kept:
# - Compute p_min from TRAIN split
# - N_test_min = ceil(20/p_min), N_test_target = 10*N_test_min
# - 20 test sizes evenly spaced between [N_test_min, N_test_target] (store min/max now)
# - N_train_min = ceil(50/p_min), N_train_max = ceil(500/p_min)
# - N_train_total = min(N_train_max, N_train_pool - ???)  -> for vision we use TRAIN split only
#   and require N_train_total > N_train_min
#
# Split policy (clean + defensible):
# - Train/Val sampled from CIFAR-10 TRAIN split (stratified)
# - Test sampled from CIFAR-10 OFFICIAL TEST split (stratified) with exact N_test_target
#
# Resolution policy: 224x224 for ALL vision datasets
# ============================================================

import numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset
from torchvision.datasets import CIFAR10
from torchvision import transforms

# ---- Use your computed values from Cell 2 ----
N_TEST_MIN = int(N_test_min)
N_TEST_TARGET = int(N_test_target)
N_TRAIN_TOTAL = int(N_train)
VAL_FRAC = 0.10
RANDOM_STATE = 42

# ----------------------------
# Load base datasets
# ----------------------------
base_train = CIFAR10(root="./data", train=True,  download=True)
y_train_full = np.array(base_train.targets, dtype=np.int64)
num_classes = int(len(np.unique(y_train_full)))

base_test = CIFAR10(root="./data", train=False, download=True)
y_test_full = np.array(base_test.targets, dtype=np.int64)

# ----------------------------
# Train/Val selection from TRAIN split (exact N_TRAIN_TOTAL stratified)
# Edge-case fix kept: if N_TRAIN_TOTAL == len(pool), don't second-split
# ----------------------------
idx_train_pool = np.arange(len(y_train_full))

if N_TRAIN_TOTAL > len(idx_train_pool):
    raise RuntimeError(f"N_TRAIN_TOTAL={N_TRAIN_TOTAL} exceeds CIFAR-10 train size={len(idx_train_pool)}")

if N_TRAIN_TOTAL == len(idx_train_pool):
    idx_trainval = idx_train_pool
else:
    idx_trainval, _ = train_test_split(
        idx_train_pool,
        train_size=N_TRAIN_TOTAL,
        stratify=y_train_full,
        random_state=RANDOM_STATE
    )

y_trainval = y_train_full[idx_trainval]

idx_train, idx_val = train_test_split(
    idx_trainval,
    test_size=VAL_FRAC,
    stratify=y_trainval,
    random_state=RANDOM_STATE
)

# ----------------------------
# Test selection from OFFICIAL TEST split (exact N_TEST_TARGET stratified)
# ----------------------------
idx_test_pool = np.arange(len(y_test_full))

if N_TEST_TARGET > len(idx_test_pool):
    raise RuntimeError(f"N_TEST_TARGET={N_TEST_TARGET} exceeds CIFAR-10 test size={len(idx_test_pool)}")

idx_test, _ = train_test_split(
    idx_test_pool,
    train_size=N_TEST_TARGET,
    stratify=y_test_full,
    random_state=RANDOM_STATE
)

# True labels in the exact Subset order (Subset preserves list order)
y_test = y_test_full[idx_test].astype(np.int64)

# ----------------------------
# Define FIXED transforms (your protocol)
# ----------------------------
IMG_SIZE = 224

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406),
                         std=(0.229, 0.224, 0.225)),
])

eval_tf = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406),
                         std=(0.229, 0.224, 0.225)),
])

# ----------------------------
# Wrap subsets with transforms
# (We instantiate datasets with different transforms; Subset picks indices.)
# ----------------------------
ds_train = CIFAR10(root="./data", train=True,  download=False, transform=train_tf)
ds_val   = CIFAR10(root="./data", train=True,  download=False, transform=eval_tf)
ds_test  = CIFAR10(root="./data", train=False, download=False, transform=eval_tf)

train_set = Subset(ds_train, idx_train.tolist())
val_set   = Subset(ds_val,   idx_val.tolist())
test_set  = Subset(ds_test,  idx_test.tolist())

# ----------------------------
# Quick reporting
# ----------------------------
def bincount_dict_from_idx(idx, y_arr, k):
    bc = np.bincount(y_arr[idx], minlength=k)
    return {int(i): int(bc[i]) for i in range(k)}

print("=== SPLITS ===")
print("Dataset: CIFAR-10")
print(f"Train+Val: {len(idx_trainval)} (target {N_TRAIN_TOTAL}) from TRAIN split")
print(f"  Train:   {len(idx_train)}")
print(f"  Val:     {len(idx_val)} (10% of train+val)")
print(f"Test:      {len(idx_test)} (target {N_TEST_TARGET}) from OFFICIAL TEST split")
print("Class counts:")
print("  Train:", bincount_dict_from_idx(idx_train, y_train_full, num_classes))
print("  Val:  ", bincount_dict_from_idx(idx_val,   y_train_full, num_classes))
print("  Test: ", bincount_dict_from_idx(idx_test,  y_test_full,  num_classes))
print()

print("=== TEST SIZE RANGE (FOR LATER 20 SIZES) ===")
print(f"N_test_min    = {N_TEST_MIN}")
print(f"N_test_target = {N_TEST_TARGET}")
print("Later you will create 20 evenly spaced sizes between these two.")
print()

print("=== TRANSFORMS ===")
print("Train: RandomResizedCrop(224) + RandomHorizontalFlip + ImageNet Normalize")
print("Eval:  Resize(224) + CenterCrop(224) + ImageNet Normalize")
print()

# Sanity check sample tensor shape
x0, y0 = train_set[0]
print("Sample tensor shape:", tuple(x0.shape), "| label:", int(y0))


In [ ]:
# ============================================================
# TRAINING CELL (Vision Pool v1) — CIFAR-10 (FIXED + SAFE GUARDS)
# Protocol:
# - Stage 1: head-only, 3 epochs, LR=1e-3
# - Stage 2: full unfreeze, 2 epochs (Policy A)
#     backbone LR=1e-5, head LR=1e-4
# - Optimizer: AdamW, weight_decay=1e-4
# - Loss: class-weighted CrossEntropy (weights from TRAIN ONLY)
# - No early stopping (but restore best checkpoint by VAL macro-F1)
#
# Fixes/Guards:
# - CLEAR CUDA cache to avoid async assert confusion after an earlier failure
# - Train label extraction uses y_train_full (from Cell 3), not "y"
# - Hard checks that labels are in [0, num_classes-1] BEFORE touching CUDA loss
# - Reinit head per run
# ============================================================

import copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score

# ----------------------------
# IMPORTANT: clear CUDA error state (after any device-side assert)
# If you got a device-side assert, you MUST restart runtime for full reset.
# This helps in many cases, but if it keeps happening: Runtime > Restart.
# ----------------------------
if torch.cuda.is_available():
    torch.cuda.empty_cache()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ----------------------------
# Hyperparams (fixed across models for this dataset)
# ----------------------------
BATCH_SIZE = 64     # set once per dataset; keep same for all 15 models on this dataset
NUM_WORKERS = 2

EPOCHS_STAGE1 = 3
EPOCHS_STAGE2 = 2

LR_HEAD_STAGE1 = 1e-3
LR_BACKBONE_STAGE2 = 1e-5
LR_HEAD_STAGE2 = 1e-4

WEIGHT_DECAY = 1e-4

# ----------------------------
# DataLoaders
# ----------------------------
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

# ----------------------------
# Class weights from TRAIN ONLY (FAST, NO image loading)
# Assumes Cell 3 created: y_train_full (np array for CIFAR train split) + idx_train
# ----------------------------
train_labels = np.asarray(y_train_full, dtype=np.int64)[np.asarray(idx_train, dtype=np.int64)]

# num_classes should match CIFAR-10 = 10, but we validate it anyway
num_classes = int(num_classes)

# ----------------------------
# HARD GUARDS: label validity (this is the #1 cause of CUDA asserts)
# ----------------------------
min_y = int(train_labels.min())
max_y = int(train_labels.max())
uniq = np.unique(train_labels)

print("=== LABEL SANITY ===")
print(f"num_classes = {num_classes}")
print(f"train_labels min/max = {min_y}/{max_y}")
print(f"unique labels (train) = {uniq.tolist()}")

if min_y < 0 or max_y >= num_classes:
    raise RuntimeError(
        f"Found invalid labels for CrossEntropyLoss: min={min_y}, max={max_y}, "
        f"but num_classes={num_classes}. Fix label mapping BEFORE training."
    )

# Also validate that the dataset is actually returning labels in-range
# (this catches weird transform/dataset wrappers)
x_tmp, y_tmp = train_set[0]
if not (0 <= int(y_tmp) < num_classes):
    raise RuntimeError(f"train_set[0] returned label {int(y_tmp)} outside [0, {num_classes-1}]")

# ----------------------------
# Compute class weights (TRAIN ONLY)
# ----------------------------
class_counts = np.bincount(train_labels, minlength=num_classes).astype(np.float64)
class_weights = (class_counts.sum() / (num_classes * np.maximum(class_counts, 1.0)))
class_weights_t = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)

print("\n=== CLASS IMBALANCE HANDLING (TRAIN ONLY) ===")
print(f"Class counts (train): {class_counts.tolist()}")
print(f"Class weights:        {class_weights.tolist()}")
print()

# ----------------------------
# Helpers: identify head module + head params
# ----------------------------
def get_head_module(model: nn.Module) -> nn.Module:
    if hasattr(model, "fc") and isinstance(model.fc, nn.Module):
        return model.fc
    if hasattr(model, "classifier") and isinstance(model.classifier, nn.Module):
        return model.classifier
    if hasattr(model, "head") and isinstance(model.head, nn.Module):
        return model.head
    if hasattr(model, "heads") and isinstance(model.heads, nn.Module):
        return model.heads
    if hasattr(model, "get_classifier"):
        clf = model.get_classifier()
        if isinstance(clf, nn.Module):
            return clf
    raise RuntimeError("Cannot identify classifier head module for this model.")

def get_head_params(model: nn.Module):
    return list(get_head_module(model).parameters())

def freeze_backbone(model: nn.Module):
    for p in model.parameters():
        p.requires_grad = False
    for p in get_head_params(model):
        p.requires_grad = True

def unfreeze_all(model: nn.Module):
    for p in model.parameters():
        p.requires_grad = True

def make_optimizer_stage1(model: nn.Module):
    return torch.optim.AdamW(get_head_params(model), lr=LR_HEAD_STAGE1, weight_decay=WEIGHT_DECAY)

def make_optimizer_stage2(model: nn.Module):
    head_params = set(id(p) for p in get_head_params(model))
    backbone = [p for p in model.parameters() if id(p) not in head_params]
    head = get_head_params(model)
    return torch.optim.AdamW(
        [
            {"params": backbone, "lr": LR_BACKBONE_STAGE2, "weight_decay": WEIGHT_DECAY},
            {"params": head,     "lr": LR_HEAD_STAGE2,     "weight_decay": WEIGHT_DECAY},
        ]
    )

# ----------------------------
# Reinitialize head per training run (deterministic)
# ----------------------------
def reset_head_weights(model: nn.Module, seed: int = 12345):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    head = get_head_module(model)

    def _init_linear(m):
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None:
                nn.init.zeros_(m.bias)

    head.apply(_init_linear)

# ----------------------------
# Train/Eval helpers
# ----------------------------
criterion = nn.CrossEntropyLoss(weight=class_weights_t)

@torch.no_grad()
def eval_val_macro_f1(model: nn.Module):
    model.eval()
    preds, trues = [], []
    for xb, yb in val_loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)
        logits = model(xb)
        preds.append(logits.argmax(dim=1).detach().cpu().numpy())
        trues.append(yb.detach().cpu().numpy())
    preds = np.concatenate(preds)
    trues = np.concatenate(trues)
    return float(f1_score(trues, preds, average="macro"))

def train_one_epoch(model: nn.Module, optimizer, criterion):
    model.train()
    for xb, yb in train_loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)

        # extra guard: catch invalid labels early on CPU-friendly error
        if int(yb.min()) < 0 or int(yb.max()) >= num_classes:
            raise RuntimeError(
                f"Batch has invalid labels: min={int(yb.min())}, max={int(yb.max())}, num_classes={num_classes}"
            )

        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

# ----------------------------
# Train all models
# ----------------------------
trained_models = {}
train_logs = {}

for name, base_model in models.items():
    print(f"\n==============================")
    print(f"[TRAIN] {name}")
    print(f"==============================")

    model = copy.deepcopy(base_model).to(DEVICE)

    # FIX: reset head init per run
    reset_head_weights(model, seed=12345)

    best_f1 = -1.0
    best_state = None

    # Stage 1: head-only
    freeze_backbone(model)
    opt1 = make_optimizer_stage1(model)

    for ep in range(1, EPOCHS_STAGE1 + 1):
        train_one_epoch(model, opt1, criterion)
        f1 = eval_val_macro_f1(model)
        print(f"  Stage1 | epoch {ep}/{EPOCHS_STAGE1} | val_macro_f1={f1:.4f} | best={best_f1:.4f}")
        if f1 > best_f1 + 1e-6:
            best_f1 = f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    # Stage 2: full unfreeze (Policy A)
    unfreeze_all(model)
    opt2 = make_optimizer_stage2(model)

    for ep in range(1, EPOCHS_STAGE2 + 1):
        train_one_epoch(model, opt2, criterion)
        f1 = eval_val_macro_f1(model)
        print(f"  Stage2 | epoch {ep}/{EPOCHS_STAGE2} | val_macro_f1={f1:.4f} | best={best_f1:.4f}")
        if f1 > best_f1 + 1e-6:
            best_f1 = f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    trained_models[name] = model
    train_logs[name] = {"best_val_macro_f1": float(best_f1)}

print("\n=== VISION TRAINING SUMMARY ===")
print(f"Trained models: {len(trained_models)}/{len(models)}")
for k, v in train_logs.items():
    print(f"  {k:18s}  best_val_macro_f1={v['best_val_macro_f1']:.4f}")

print("\nDONE. (No test prediction / no evaluation in this cell.)")


In [ ]:
# ============================================================
# VISION CELL 5 — EVAL + PRINT PERFORMANCE + SAVE JSON TO DRIVE (ADAPTED)
# Adds:
# - N_test_min, N_test_target stored in JSON (so you never type manually)
# - alignment assertion: y_test order matches test_set order
# ============================================================

import os, json
import numpy as np
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score

# ----------------------------
# Paths (edit BASE_DIR if needed)
# ----------------------------


DATASET_NAME = "CIFAR10"
BASE_DIR = "data/predictions"
OUT_DIR = os.path.join(BASE_DIR, DATASET_NAME)
os.makedirs(OUT_DIR, exist_ok=True)

OUT_JSON = os.path.join(OUT_DIR, f"{DATASET_NAME}_test_predictions_logits.json")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ----------------------------
# DataLoader (test)
# ----------------------------
BATCH_SIZE_EVAL = 128
NUM_WORKERS = 2
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE_EVAL, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

# ----------------------------
# Alignment assertion (must hold)
# ----------------------------
y_test_np = np.asarray(y_test, dtype=np.int64)

y_check = []
for i in range(len(test_set)):
    _, yi = test_set[i]
    y_check.append(int(yi))
y_check = np.array(y_check, dtype=np.int64)

assert np.array_equal(y_check, y_test_np), "Mismatch: y_test order != test_set order"

# ----------------------------
# Eval helper
# ----------------------------
@torch.no_grad()
def predict_logits(model):
    model.eval()
    chunks = []
    for xb, _ in test_loader:
        xb = xb.to(DEVICE, non_blocking=True)
        logits = model(xb).detach().cpu().numpy()
        chunks.append(logits)
    return np.concatenate(chunks, axis=0)

# ----------------------------
# Evaluate + collect
# ----------------------------
results = {
    "dataset": DATASET_NAME,
    "num_classes": int(num_classes),

    # ✅ store your test size range so downstream notebooks don't need manual input
    "N_test_min": int(N_test_min),
    "N_test_target": int(N_test_target),

    # ✅ true labels included
    "y_test": y_test_np.tolist(),

    "models": {}
}

print("=== TEST PERFORMANCE (accuracy, macro_f1) ===")

for name, model in trained_models.items():
    logits = predict_logits(model)
    pred = logits.argmax(axis=1)

    acc = float(accuracy_score(y_test_np, pred))
    mf1 = float(f1_score(y_test_np, pred, average="macro"))

    print(f"{name:18s}  acc={acc:.4f}  macro_f1={mf1:.4f}")

    results["models"][name] = {
        "logits": logits.tolist(),   # raw discriminant outputs (NO softmax)
        "y_pred": pred.tolist(),
        "metrics": {"accuracy": acc, "macro_f1": mf1},
    }

print("\nModels evaluated:", len(results["models"]))
print("Saving to:", OUT_JSON)

with open(OUT_JSON, "w") as f:
    json.dump(results, f)

print("✅ Saved.")
print("✅ True labels are included in JSON under key: 'y_test'")
print("✅ Test size range included in JSON under keys: 'N_test_min', 'N_test_target'")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# VISION CELL 6 — PAIRWISE DISAGREEMENT (LABELS)
#
# Assumes you have ONE of these in memory:
#  1) results (dict) from Cell 5
#  2) loaded_json (dict) if you manually loaded the saved JSON file
#  3) all_predictions_pred[name] = np.array([...])
#  4) all_predictions_logits[name] = np.array([N_test, C])  -> will argmax to y_pred
#
# Decision (recommended now):
# - Use full test size: N = N_test_target
# ============================================================

# ---- Resolve predictions dict ----
if "all_predictions_pred" in globals():
    pred_dict = {k: np.asarray(v) for k, v in all_predictions_pred.items()}
    meta = {}
elif "results" in globals() and isinstance(results, dict) and "models" in results:
    pred_dict = {k: np.asarray(v["y_pred"]) for k, v in results["models"].items()}
    meta = results
elif "loaded_json" in globals() and isinstance(loaded_json, dict) and "models" in loaded_json:
    pred_dict = {k: np.asarray(v["y_pred"]) for k, v in loaded_json["models"].items()}
    meta = loaded_json
elif "all_predictions_logits" in globals():
    pred_dict = {k: np.asarray(v).argmax(axis=1) for k, v in all_predictions_logits.items()}
    meta = {}
else:
    raise RuntimeError(
        "No predictions found. Expected all_predictions_pred, or results/loaded_json['models'][..]['y_pred'], "
        "or all_predictions_logits."
    )

# ---- Decide N to use (default: N_test_target if present in meta) ----
N_use = None
if isinstance(meta, dict) and ("N_test_target" in meta):
    N_use = int(meta["N_test_target"])

# If you want to override manually, uncomment:
# N_use = 2000

# ---- Validate same length across models + optionally slice to N_use ----
model_names = sorted(pred_dict.keys())
lengths = {m: int(len(pred_dict[m])) for m in model_names}
N0 = list(lengths.values())[0]
if any(n != N0 for n in lengths.values()):
    raise RuntimeError(f"Not all models have same N. Lengths: {lengths}")

if N_use is None:
    N_use = N0
if N_use > N0:
    raise RuntimeError(f"N_use={N_use} exceeds available N={N0}")

preds = np.stack([pred_dict[m][:N_use] for m in model_names], axis=0)  # (M, N_use)

M, N = preds.shape
disagree = np.zeros((M, M), dtype=np.float64)

# ============================================================
# 1) Pairwise disagreement matrix (fraction different labels)
# ============================================================
for i in range(M):
    for j in range(M):
        disagree[i, j] = np.mean(preds[i] != preds[j])

# ============================================================
# 2) Off-diagonal summary stats
# ============================================================
off_mask = ~np.eye(M, dtype=bool)
off_vals = disagree[off_mask]

print("=== Off-diagonal disagreement stats ===")
print(f"num_models: {M}, num_samples_used: {N}")
print(f"mean: {off_vals.mean():.6f}")
print(f"std:  {off_vals.std(ddof=0):.6f}")
print(f"min:  {off_vals.min():.6f}")
print(f"max:  {off_vals.max():.6f}")

# ============================================================
# 3) Plots
#  - heatmap of disagreement matrix
#  - histogram of off-diagonal disagreement values
# ============================================================
plt.figure(figsize=(max(8, 0.45 * M), max(6, 0.45 * M)))
plt.imshow(disagree, aspect="auto")
plt.colorbar(label="Disagreement (fraction different labels)")
plt.xticks(range(M), model_names, rotation=90)
plt.yticks(range(M), model_names)
title_suffix = f"(N={N})"
plt.title(f"Pairwise Disagreement Matrix {title_suffix}")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4.5))
plt.hist(off_vals, bins=30)
plt.title(f"Histogram of Off-diagonal Pairwise Disagreements (N={N})")
plt.xlabel("Disagreement (fraction different labels)")
plt.ylabel("Count")
plt.tight_layout()
plt.show()
